# realworld2.ipynb

This notebook was somewhat inspired by a Medium article I recently read which suggests using SQL instead of pandas for increased efficiency.

I will try to rewrite parts of realworld.ipynb to try out this concept.

I already figured out how to store numpy arrays in SQLite without converting to text, by using WKT for example. See `np2sqlite.py`.

In [1]:
from ultralytics.models.sam import SAM3SemanticPredictor
from np2sqlite import array2blob, blob2array
# from roadside import test_build_db, get_config
import numpy as np
from icecream import ic
import os
import sqlite3
import cv2
import gc
import torch

from pyefd import elliptic_fourier_descriptors, reconstruct_contour
# from shapely.wkt import loads

# import pandas as pd
import exif
import tomli as tomllib
from glob import glob

# Functions

In [2]:
def run_sam3_semantic_predictor(input_image_path: str, text_prompts: list=['coconut palm tree']) -> list:
    """ 
    Uses the SAM3 semantic predictor to detect objects specified by text prompts in an image.
    
    Inputs:
      input_image_path relative to working directory 
      text_prompts: list of text prompts; default: ['coconut palm tree']
      
    Outputs:
      results:     
    """
    # Initialize predictor with configuration
    overrides = dict(
        conf=0.25,
        task="segment",
        mode="predict",
        model="sam3.pt",
        half=True,  # Use FP16 for faster inference
        save=True,  # Save image visualizing output results
        save_txt=False,  # Save output results in text format
        save_conf=False,  # Save confidence scores   
        imgsz=1932,  # Adjusted image size from 1920 to meet stride 14 requirement
        batch=1,
        device="0",  # Use GPU device 0
    )
    predictor = SAM3SemanticPredictor(overrides=overrides)

    # Set image once for multiple queries
    predictor.set_image(input_image_path)

    # Query with multiple text prompts
    results = predictor(text=text_prompts)

    return results

## Example usage:

# root_dir = "/home/aubrey/Desktop/sam3-2026-01-31"
# image_paths = ["20251129_152106.jpg", "08hs-palms-03-zglw-superJumbo.webp"]
# text_prompts = ["coconut palm tree"]

# os.chdir(root_dir) # ensure we start in the correct directory
# for image_path in image_paths:
#     results_gpu = run_sam3_semantic_predictor(image_path, text_prompts)

#     # Free up GPU memory in preparation for detecting objects in the next image
#     # This is a work-around to prevent out-of-memory errors from the GPU
#     # I move all results for further processing and use the GPU only for object detection.
#     print('deleting results from GPU memory')       
#     results_cpu = [r.cpu() for r in results_gpu] # copy results to CPU
#     delete_results_from_gpu_memory()

# print("Processing complete.")


In [ ]:
def build_db(db_path, image_paths, schema_sql) -> None:

    conn = sqlite3.connect(db_path)   
    conn.enable_load_extension(True)
    conn.load_extension('mod_spatialite')
    conn.execute("SELECT InitSpatialMetaData(1);")
    conn.executescript(schema_sql)
    conn.commit()
          
    for image_path in image_paths:
        # ensure GPU memory is empty before processing image
        # del results_gpu
        if "model" in globals():
            print('deleting model from GPU memory')
            del model
        gc.collect()
        torch.cuda.empty_cache()
            
        # run the SAM3 semantic predictor on an image and move results to CPU for further processing
        results_gpu = run_sam3_semantic_predictor(
            input_image_path=image_path, 
            text_prompts=["coconut palm tree"]
        )
        
        # Free up GPU memory in preparation for detecting objects in the next image
        # This is a work-around to prevent out-of-memory errors from the GPU
        # I move all results for further processing and use the GPU only for object detection.
        print('copying results_gpu to results_cpu')
        results_cpu = [r.cpu() for r in results_gpu] # copy results to CPU
        print('deleting results_gpu from GPU')       
        del results_gpu 
        gc.collect() 
        torch.cuda.empty_cache() # Clears unoccupied cached memory
        
        # add record to images table
        #############################
        image_height = results_cpu[0].orig_shape[0]
        image_width = results_cpu[0].orig_shape[1]
        image_cursor = conn.execute(
            "INSERT INTO images (image_path, image_width, image_height) VALUES (?, ?, ?)", 
            (image_path, image_width, image_height)
        )
        image_id = image_cursor.lastrowid
        conn.commit()
        
        with open(image_path, 'rb') as f:
            imgx = exif.Image(f)
            if imgx.has_exif:
                # timestamp
                timestamp = imgx.datetime
                    
                # latitude
                d, m, s = imgx.gps_latitude
                latitude = d + m/60 + s/3600   
                if imgx.gps_latitude_ref == 'S':
                    latitude = -latitude              

                # longitude
                d, m, s = imgx.gps_longitude
                longitude = d + m/60 + s/3600   
                if imgx.gps_longitude_ref == 'W':
                    longitude = -longitude
                longitude

                wkt = f'POINT ({longitude} {latitude})'
                
                conn.execute(
                    "UPDATE images SET timestamp = ?, location = GeomFromText(?, 4326) WHERE image_path = ?",
                    (timestamp, wkt, image_path)
                )
                conn.commit()
                            
        # add records to trees table
        #################################
        
        cpu_results = results_cpu # FIX THIS
        boxes = cpu_results[0].boxes
        conf_list = boxes.conf.cpu().numpy().tolist()
        class_list = boxes.cls.cpu().numpy().tolist()
        # tree_contour_list = cpu_results[0].masks.xy

        try:
            # 1. Access your raw mask tensor from the Ultralytics Masks object
            # (Assuming `results[0].masks.data[0]` is your torch.Tensor of sha
            mask_tensor = cpu_results[0].masks.data
        except AttributeError:
            continue

        # 2. Convert PyTorch Tensor -> NumPy array
        # We move it to CPU, convert to numpy, and cast to uint8 (0 and 255)
        binary_masks = (mask_tensor.cpu().numpy() * 255).astype(np.uint8)
        # ic(binary_masks);
        # cv2.imwrite('binary_mask.png', binary_mask)

        for i, binary_mask in enumerate(binary_masks):
            # cv2.imwrite(f'binary_mask_{i}.png', binary_mask)

            # 3. Find clean, isolated contours
            # RETR_EXTERNAL ignores internal holes and fragment hierarchies completely
            # CHAIN_APPROX_SIMPLE implements lossless compression by removing coordinates on straight lines
            contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            tree_contour = max(contours, key=cv2.contourArea)

            # 5. Draw them cleanly on your original image
            # -1 draws all found contours; (0, 255, 0) is green; 2 is the line thickness
            # output_image = cpu_results[0].orig_img.copy()  # Grab the original BGR image from Ultralytics
            # cv2.drawContours(output_image, [largest_contour], -1, (0, 255, 0), 2)
            # cv2.imwrite(f'output_image_{i}.png', output_image)
            
            # 1. Ensure it's a mutable NumPy array or list
            # YOLO .xy returns a float32 numpy array
            if len(tree_contour) == 0:
                continue
            
            # convert tree_contour to int32 numpy array
            tree_contour = tree_contour.astype(np.int32)
            tree_contour = np.squeeze(tree_contour)

            # 2. Check if the last coordinate matches the first
            # tree_contour[0] is first point [x, y], tree_contour[-1] is last point [x, y]
            if not np.array_equal(tree_contour[0], tree_contour[-1]):
                # Append the first point to the end to close the loop
                tree_contour = np.vstack([tree_contour, tree_contour[0]])  
                                         
            # convert tree_contour from np.int32 to WKT   
            coord_str = ', '.join([f'{coord[0]} {coord[1]}' for coord in tree_contour])
            wkt = f"POLYGON (({coord_str}))"
            
            # 4. Insert into the database
            
            class_id = class_list[i]
            confidence = conf_list[i]
            detection_cursor = conn.execute(
                "INSERT INTO trees (image_id, class_id, confidence, poly) VALUES (?, ?, ?, GeomFromText(?, 0))", 
                (image_id, class_id, confidence, wkt)
            )
            conn.commit()

            # add records to damage table
            #############################  
            
            detection_id = detection_cursor.lastrowid 
            ic(detection_id)
            defect_contours = calc_defect_contours(image_height, image_width, tree_contour, config['order'], config['minpixels'])  
            for defect_contour in defect_contours: 
                defect_contour = np.squeeze(defect_contour)         
                # convert defect_contour from np.int32 to WKT 
                coord_str = ', '.join([f'{coord[0]} {coord[1]}' for coord in defect_contour])
                wkt = f"POLYGON (({coord_str}))"
                conn.execute(
                    'INSERT INTO damage (image_id, detection_id, poly) VALUES (?, GeomFromText(?, 0))', 
                    (image_id, detection_id, wkt)
                )
            conn.commit() 
            
    # postprocess database
    ######################
    
    # flip ys in tree contours so they overlay images
    sql = 'UPDATE trees SET poly = ATM_Transform(poly, ATM_CreateScale(1, -1));'
    conn.execute(sql)
    conn.commit()
    
    # flip ys in damage contours so they overlay images
    sql = 'UPDATE damage SET poly = ATM_Transform(poly, ATM_CreateScale(1, -1));'
    conn.execute(sql)
    conn.commit()

    conn.close()
        
        
def test_build_db():
    os.remove(config['dbpath']) if os.path.exists(config['dbpath']) else None
    build_db(
        db_path = config['dbpath'], 
        image_paths = [
            'data_cache/example_images/20251129_152106.jpg',
            'data_cache/example_images/data_cache/example_images/08hs-palms-03-zglw-superJumbo.webp',
            ], 
        schema_sql = config['default_schema_sql']
        )
 
# test_build_db()

In [4]:
def reconstruct_aligned_mask(image_shape, contour, order=10, align_to_centroid=True):
    """
    Finds EFDs and reconstructs the mask perfectly aligned with the original locus.
    
    Parameters:
        image_shape (tuple): Shape of the original image (H, W)
        contour (ndarray): Contour array of original image; shape (N, 2) or (N, 1, 2)
        order (int): Number of Fourier coefficients to use
        
    Returns:
        ndarray: Binary mask with the reconstructed shape in the correct position
    """
    
    ic()
    
    # 1. Standardize contour shape to (N, 2)
    contour = contour.reshape(-1, 2)
    
    # 2. Calculate the true centroid (locus) of the original contour using moments.
    # This keeps the reconstructed shape strictly bound to the true defect location.
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cX = M["m10"] / M["m00"]
        cY = M["m01"] / M["m00"]
    else:
        cX, cY = np.mean(contour, axis=0)

    # 3. Compute EFD coefficients (keeping unnormalized to retain spatial properties)
    coeffs = elliptic_fourier_descriptors(contour, order=order, normalize=False)
    
    # 4. Corrected function: Reconstruct contour points via the native API.
    # We pass the calculated cX, cY into the locus argument.
    # Next line added by Aubrey Moore 2026-06-02
    num_points = contour.shape[0]  # Use the original number of contour points for reconstruction
    reconstructed_points = reconstruct_contour(coeffs, locus=(cX, cY), num_points=num_points)
    
    # 5. Prevent sub-pixel "floor bias" shift by rounding before converting to integer
    reconstructed_contour = np.round(reconstructed_points).astype(np.int32)
    reconstructed_contour = reconstructed_contour.reshape(-1, 1, 2)
    
    # 6. Create the aligned mask
    reconstructed_mask = np.zeros(image_shape, dtype=np.uint8)
    cv2.drawContours(reconstructed_mask, [reconstructed_contour], -1, 255, -1)
    
    if align_to_centroid:
        # Calculate the centroid of the reconstructed mask
        M_recon = cv2.moments(reconstructed_contour)
        if M_recon["m00"] != 0:
            recon_cX = M_recon["m10"] / M_recon["m00"]
            recon_cY = M_recon["m01"] / M_recon["m00"]
        else:
            recon_cX, recon_cY = np.mean(reconstructed_contour.reshape(-1, 2), axis=0)
        
        # Calculate the shift needed to align the reconstructed contour's centroid with the original
        shift_x = int(cX - recon_cX)
        shift_y = int(cY - recon_cY)
        ic(shift_x, shift_y)
        
        # Shift the reconstructed contour and mask
        translation_matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
        reconstructed_mask = cv2.warpAffine(reconstructed_mask, translation_matrix, (image_shape[1], image_shape[0]))
        reconstructed_contour = cv2.transform(reconstructed_contour, translation_matrix)
    
    return reconstructed_contour, reconstructed_mask


In [5]:
def get_centroid(contour):
    """ Returns centroid of a contour. """
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
    else:
        cX, cY = np.mean(contour, axis=0)
    return cX, cY    

In [6]:
def calc_defect_contours(image_height, image_width, tree_contour, order, minpixels):     
    canvas = np.zeros((image_height, image_width), np.uint8)
    tree_mask = cv2.drawContours(canvas, [tree_contour], -1, 255, -1)
    tree_mask_cx, tree_mask_cy = get_centroid(tree_mask)
    
    _, reconstructed_mask = reconstruct_aligned_mask(image_shape=(image_height, image_width), contour=tree_contour, order=order)
    # reconstructed_mask_cx, reconstructed_mask_cy = get_centroid(reconstructed_mask)
    registered_mask = reconstructed_mask.copy()
    additions_mask = cv2.bitwise_and(registered_mask, cv2.bitwise_not(tree_mask))    
    defect_contours, _ = cv2.findContours(additions_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    defect_contours = [cnt for cnt in defect_contours if cv2.contourArea(cnt) > minpixels]

    return defect_contours

In [7]:
def save_sam_results_to_spatialite(masks, scores, labels=None, default_label="object"):
    """
    Parses raw SAM3 binary prediction masks into pixel-space MultiPolygons 
    and saves them to the initialized SpatiaLite database.
    
    Parameters:
        masks (list or np.array): List/array of boolean/binary masks from SAM3.
        scores (list or np.array): Confidence scores corresponding to each mask.
        labels (list, optional): Text labels for each mask. Defaults to generic naming.
        default_label (str): Fallback label if 'labels' array isn't provided.
    """
    print(f"Starting conversion for {len(masks)} prediction masks...")
    inserted_count = 0

    for idx, (mask, score) in enumerate(zip(masks, scores)):
        # 1. Convert boolean matrix to uint8 image (0 or 255)
        # OpenCV naturally treats the top-left index of this array as (0,0)
        binary_mask = (mask.astype(np.uint8)) * 255
        
        # 2. Find external contours
        # RETR_EXTERNAL keeps outermost boundaries; CHAIN_APPROX_SIMPLE compresses segments
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        polygon_strings = []
        
        for contour in contours:
            # Reshape OpenCV array from (N, 1, 2) to standard vertex coordinate list (N, 2)
            points = contour.reshape(-1, 2)
            
            # OGC Standards require a valid polygon shell to contain at least 3 distinct vertices
            if len(points) >= 3:
                # Ensure the polygon closes perfectly by appending the first point to the end if missing
                if not np.array_equal(points[0], points[-1]):
                    points = np.vstack([points, points[0]])
                
                # Format vertices into "X Y" space strings
                point_strs = [f"{pt[0]} {pt[1]}" for pt in points]
                polygon_strings.append(f"(({', '.join(point_strs)}))")
                
        if not polygon_strings:
            # Skip empty masks where no physical contours could be wrapped
            continue
            
        # 3. Construct Well-Known Text (WKT) representation
        multipolygon_wkt = f"MULTIPOLYGON({', '.join(polygon_strings)})"
        
        # 4. Resolve labels
        current_label = labels[idx] if labels else f"{default_label}_{idx}"
        
        # 5. Insert to SpatiaLite Using GeomFromText
        srid = 0
        cursor.execute(f"""
        INSERT INTO mask_contours (mask_index, label, confidence, geom)
        VALUES (?, ?, ?, GeomFromText(?, {srid}))
        """, (idx, current_label, float(score), multipolygon_wkt))
        
        inserted_count += 1

    # Save changes to disk
    conn.commit()
    print(f"Successfully processed and stored {inserted_count} features into '{db_path}'.")

# MAIN

In [8]:
with open("config.toml", mode="rb") as f:
        config = tomllib.load(f)
for key, value in config.items():
    ic(key, value)

ic| key: 'default_schema_sql'
    value: '''
                -- images table
            
                CREATE TABLE IF NOT EXISTS images (
                    image_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    image_path TEXT UNIQUE,
                    image_width INTEGER,
                    image_height INTEGER,
                    timestamp TEXT
                );
            
                SELECT AddGeometryColumn(
                    'images',            -- Table name
                    'location',          -- Column name
                    4326,                -- SRID (-1 or 0 signifies flat pixel/Cartesian space)
                    'POINT',             -- Geometry type
                    'XY'                 -- 2D coordinates
                );
            
                SELECT CreateSpatialIndex(
                    'trees', 
                    'poly'
                );
            
                -- trees table
            
                CREATE TAB

In [9]:
# build a new database

db_path = '/home/aubrey/Desktop/Efate2025/Efate2025A.db'
os.remove(db_path) if os.path.exists(db_path) else None

image_paths = glob('/home/aubrey/Desktop/Efate2025/original_images/*.jpg')
n = 100
image_paths = image_paths[::n] # every nth image_path
ic(len(image_paths))

schema_sql = config['default_schema_sql']

build_db(db_path, image_paths, schema_sql)

ic| len(image_paths): 40
CreateSpatialIndex() error: either "trees"."poly" isn't a Geometry column or a SpatialIndex is already defined


Ultralytics 8.4.49 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)

image 1/1 /home/aubrey/Desktop/Efate2025/original_images/20251129_150658.jpg: 1932x1932 3 coconut palm trees, 1430.2ms
Speed: 17.8ms preprocess, 1430.2ms inference, 31.8ms postprocess per image at shape (1, 3, 1932, 1932)
Results saved to /home/aubrey/Desktop/blog2026/runs/segment/predict-152
copying results_gpu to results_cpu
deleting results_gpu from GPU


ic| detection_id: 1
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:32:15.671
ic| shift_x: -16, shift_y: 45
ic| detection_id: 2
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:32:15.743
ic| shift_x: -5, shift_y: 23
ic| detection_id: 3
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:32:15.783
ic| shift_x: -2, shift_y: 58


Ultralytics 8.4.49 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)

image 1/1 /home/aubrey/Desktop/Efate2025/original_images/20251127_183227.jpg: 1932x1932 (no detections), 1299.5ms
Speed: 17.7ms preprocess, 1299.5ms inference, 0.7ms postprocess per image at shape (1, 3, 1932, 1932)
Results saved to /home/aubrey/Desktop/blog2026/runs/segment/predict-153
copying results_gpu to results_cpu
deleting results_gpu from GPU
Ultralytics 8.4.49 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)

image 1/1 /home/aubrey/Desktop/Efate2025/original_images/20251127_175435.jpg: 1932x1932 (no detections), 1304.3ms
Speed: 15.3ms preprocess, 1304.3ms inference, 1.3ms postprocess per image at shape (1, 3, 1932, 1932)
Results saved to /home/aubrey/Desktop/blog2026/runs/segment/predict-154
copying results_gpu to results_cpu
deleting results_gpu from GPU
Ultralytics 8.4.49 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeFo

ic| detection_id: 4
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:32:49.908
ic| shift_x: -16, shift_y: 16
ic| detection_id: 5
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:32:49.971
ic| shift_x: -5, shift_y: -63
ic| detection_id: 6
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:32:50.033
ic| shift_x: -1, shift_y: 1
ic| detection_id: 7
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:32:50.075
ic| shift_x: 0, shift_y: 0


...

ic| detection_id: 124
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:01.055
ic| shift_x: -11, shift_y: -13
ic| detection_id: 125
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:01.118
ic| shift_x: 0, shift_y: 0
ic| detection_id: 126
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:01.159
ic| shift_x: 1, shift_y: 0
ic| detection_id: 127
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:01.201
ic| shift_x: -8, shift_y: 5
ic| detection_id: 128
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:01.243
ic| shift_x: -3, shift_y: 2


Ultralytics 8.4.49 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)

image 1/1 /home/aubrey/Desktop/Efate2025/original_images/20251127_174016.jpg: 1932x1932 9 coconut palm trees, 1332.8ms
Speed: 15.5ms preprocess, 1332.8ms inference, 2.8ms postprocess per image at shape (1, 3, 1932, 1932)
Results saved to /home/aubrey/Desktop/blog2026/runs/segment/predict-190
copying results_gpu to results_cpu
deleting results_gpu from GPU


ic| detection_id: 129
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:12.098
ic| shift_x: 15, shift_y: 12
ic| detection_id: 130
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:12.159
ic| shift_x: -1, shift_y: 0
ic| detection_id: 131
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:12.200
ic| shift_x: -14, shift_y: 42
ic| detection_id: 132
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:12.263
ic| shift_x: 5, shift_y: 7
ic| detection_id: 133
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:12.330
ic| shift_x: 17, shift_y: -12
ic| detection_id: 134
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:12.396
ic| shift_x: -14, shift_y: 41
ic| detection_id: 135
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:12.462
ic| shift_x: -34, shift_y: -15
ic| detection_id: 136
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:12.526
ic| shift_x: -10, shift_y: -11
ic| detection_id: 137
ic| 3337628227.py:14 in recons

Ultralytics 8.4.49 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)

image 1/1 /home/aubrey/Desktop/Efate2025/original_images/20251127_172621.jpg: 1932x1932 5 coconut palm trees, 1329.6ms
Speed: 14.8ms preprocess, 1329.6ms inference, 1.6ms postprocess per image at shape (1, 3, 1932, 1932)
Results saved to /home/aubrey/Desktop/blog2026/runs/segment/predict-191
copying results_gpu to results_cpu
deleting results_gpu from GPU


ic| detection_id: 138
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:23.293
ic| shift_x: 0, shift_y: 11
ic| detection_id: 139
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:23.360
ic| shift_x: -26, shift_y: 10
ic| detection_id: 140
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:23.427
ic| shift_x: -4, shift_y: 10
ic| detection_id: 141
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:23.497
ic| shift_x: 24, shift_y: 20
ic| detection_id: 142
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 17:39:23.564
ic| shift_x: -4, shift_y: 41
